# 05 — Final Load Prep (Tableau-Ready Export)

**Objective:** Compute final KPIs, add team-level aggregated metrics, optimize data types, and export the Tableau-ready dataset.

**Input:** `data/processed/cleaned_worldcup_players.csv`  
**Output:** `data/processed/tableau_ready_worldcup_players.csv`

## 5.1 — Setup & Imports

In [ ]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = (
    Path.cwd().resolve().parent
    if Path.cwd().resolve().name == "notebooks"
    else Path.cwd().resolve()
)

DATA_PATH = PROJECT_ROOT / "data/processed/cleaned_worldcup_players.csv"
TABLEAU_READY_PATH = PROJECT_ROOT / "data/processed/tableau_ready_worldcup_players.csv"

print(f"Input  : {DATA_PATH}")
print(f"Output : {TABLEAU_READY_PATH}")

## 5.2 — Load Cleaned Data

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()

## 5.3 — Compute Team-Level KPIs

These metrics are computed at the team level and then merged back onto every row, so Tableau can use them as dimensions for filtering and coloring.

In [ ]:
# Team-level aggregations
team_kpi = df.groupby("team_initials").agg(
    team_total_goals=("goals", "sum"),
    team_total_yellow_cards=("yellow_cards", "sum"),
    team_total_red_cards=("red_cards", "sum"),
    team_total_penalties=("penalties_scored", "sum"),
    team_total_own_goals=("own_goals", "sum"),
    team_total_dismissals=("total_dismissals", "sum"),
    team_matches_played=("matchid", "nunique"),
    team_unique_players=("player_name", "nunique"),
    team_total_appearances=("player_name", "count")
).reset_index()

# Per-match ratios
team_kpi["team_goals_per_match"] = (
    team_kpi["team_total_goals"] / team_kpi["team_matches_played"]
).round(3)
team_kpi["team_cards_per_match"] = (
    (team_kpi["team_total_yellow_cards"] + team_kpi["team_total_red_cards"])
    / team_kpi["team_matches_played"]
).round(3)
team_kpi["team_yellows_per_match"] = (
    team_kpi["team_total_yellow_cards"] / team_kpi["team_matches_played"]
).round(3)

print(f"Team KPIs computed for {len(team_kpi)} teams.")
team_kpi.sort_values("team_total_goals", ascending=False).head(10)

## 5.4 — Merge Team KPIs onto Player-Level Data

In [ ]:
df_tableau = df.merge(team_kpi, on="team_initials", how="left")

print(f"Shape after merge: {df_tableau.shape[0]:,} rows × {df_tableau.shape[1]} columns")
print(f"Row count preserved: {len(df_tableau) == len(df)}")

## 5.5 — Add Player-Level Boolean Flags (for Tableau Filters)

In [ ]:
# Boolean flags for easy Tableau filtering
df_tableau["is_goalscorer"] = df_tableau["goals"] > 0
df_tableau["is_carded"] = (df_tableau["yellow_cards"] + df_tableau["red_cards"]) > 0
df_tableau["is_starter"] = df_tableau["line_up"] == "Starting"
df_tableau["is_substitute_used"] = df_tableau["sub_in"] == True
df_tableau["is_penalty_taker"] = (
    (df_tableau["penalties_scored"] > 0) | (df_tableau["missed_penalties"] > 0)
)
df_tableau["foreign_coach"] = df_tableau["team_initials"] != df_tableau["coach_nationality"]

flag_cols = ["is_goalscorer", "is_carded", "is_starter", "is_substitute_used",
             "is_penalty_taker", "foreign_coach"]
print("Boolean flag summary:")
for col in flag_cols:
    true_count = df_tableau[col].sum()
    print(f"  {col:25s} True: {true_count:>6,}  ({true_count / len(df_tableau) * 100:5.1f}%)")

## 5.6 — Rename Columns for Tableau Display

In [ ]:
# Create Tableau-friendly display names (Title Case, no underscores)
rename_map = {
    "roundid": "Round ID",
    "matchid": "Match ID",
    "team_initials": "Team",
    "coach_name": "Coach Name",
    "coach_nationality": "Coach Nationality",
    "line_up": "Lineup Status",
    "shirt_number": "Shirt Number",
    "player_name": "Player Name",
    "position": "Position",
    "event": "Event (Raw)",
    "is_goalkeeper": "Is Goalkeeper",
    "is_captain": "Is Captain",
    "goals": "Goals",
    "penalties_scored": "Penalties Scored",
    "own_goals": "Own Goals",
    "yellow_cards": "Yellow Cards",
    "red_cards": "Red Cards",
    "second_yellow_reds": "Second Yellow Reds",
    "missed_penalties": "Missed Penalties",
    "sub_in": "Subbed In",
    "sub_out": "Subbed Out",
    "team_total_goals": "Team Total Goals",
    "team_total_yellow_cards": "Team Total Yellow Cards",
    "team_total_red_cards": "Team Total Red Cards",
    "team_total_penalties": "Team Total Penalties",
    "team_total_own_goals": "Team Total Own Goals",
    "team_matches_played": "Team Matches Played",
    "team_unique_players": "Team Unique Players",
    "team_total_appearances": "Team Total Appearances",
    "team_goals_per_match": "Team Goals/Match",
    "team_cards_per_match": "Team Cards/Match",
    "team_yellows_per_match": "Team Yellows/Match",
    "is_goalscorer": "Is Goalscorer",
    "is_carded": "Is Carded",
    "is_starter": "Is Starter",
    "is_substitute_used": "Is Substitute Used",
    "is_penalty_taker": "Is Penalty Taker",
    "total_dismissals": "Total Dismissals",
    "foreign_coach": "Foreign Coach"
}

df_tableau = df_tableau.rename(columns=rename_map)
print(f"Columns renamed: {len(rename_map)} columns")

## 5.7 — Optimize Data Types

In [ ]:
# Convert categoricals for smaller file size and Tableau performance
cat_cols = ["Team", "Lineup Status", "Position", "Coach Nationality"]
for col in cat_cols:
    if col in df_tableau.columns:
        df_tableau[col] = df_tableau[col].astype("category")

# Ensure boolean columns are clean
bool_cols = [c for c in df_tableau.columns if c.startswith("Is ") or c == "Foreign Coach"
             or c in ["Subbed In", "Subbed Out"]]
for col in bool_cols:
    df_tableau[col] = df_tableau[col].astype(bool)

print("Data types optimized.")
print(f"\nMemory usage: {df_tableau.memory_usage(deep=True).sum() / 1e6:.2f} MB")

## 5.8 — Final Schema Review

In [ ]:
print("=" * 70)
print("TABLEAU-READY DATASET — FINAL SCHEMA")
print("=" * 70)
print(f"Rows   : {df_tableau.shape[0]:,}")
print(f"Columns: {df_tableau.shape[1]}")
print("\nColumn listing:")
for i, (col, dtype) in enumerate(df_tableau.dtypes.items(), 1):
    null_count = df_tableau[col].isna().sum()
    null_info = f" ({null_count:,} nulls)" if null_count > 0 else ""
    print(f"  {i:2d}. {col:30s} {str(dtype):15s}{null_info}")

In [ ]:
df_tableau.head(10)

## 5.9 — Export Tableau-Ready CSV

In [ ]:
TABLEAU_READY_PATH.parent.mkdir(parents=True, exist_ok=True)
df_tableau.to_csv(TABLEAU_READY_PATH, index=False)

# Verify
verify = pd.read_csv(TABLEAU_READY_PATH)
file_size_mb = TABLEAU_READY_PATH.stat().st_size / 1e6

print(f"Saved Tableau-ready dataset to:\n  {TABLEAU_READY_PATH}")
print(f"\nVerification:")
print(f"  Rows    : {verify.shape[0]:,}")
print(f"  Columns : {verify.shape[1]}")
print(f"  File size: {file_size_mb:.2f} MB")

## 5.10 — Tableau Usage Guide

### Recommended Dimensions (Filters / Color / Detail)
| Column | Use In Tableau |
|--------|----------------|
| Team | Filter, Color |
| Player Name | Detail, Label |
| Position | Color, Filter |
| Lineup Status | Filter |
| Coach Name | Detail |
| Coach Nationality | Color, Filter |
| Is Goalscorer / Is Carded / Is Captain | Filter |
| Foreign Coach | Filter |

### Recommended Measures (Axes / Size / Tooltip)
| Column | Use In Tableau |
|--------|----------------|
| Goals | Y-axis, Size, Tooltip |
| Yellow Cards / Red Cards | Y-axis, Color |
| Team Goals/Match | Y-axis (efficiency) |
| Team Cards/Match | Y-axis (discipline) |
| Team Matches Played | Size, X-axis |
| Team Unique Players | Tooltip |

### Suggested Dashboard Sheets
1. **Team Dominance** — Bar chart of Team Total Goals, colored by Team Cards/Match
2. **Player Spotlight** — Scatter of individual goals vs cards, filtered by Team
3. **Coaching Insights** — Map or bar of Coach Nationality distribution, Foreign Coach percentage

✅ **Pipeline complete. The dataset is ready for Tableau import.**